# 보건의료 교대근무자의 전신 피로 예측
### 제7차(2023) 근로환경조사 원시자료를 이용한 분류 모델

| 항목 | 내용 |
|---|---|
| **데이터** | 제7차(2023년) 근로환경조사 원시자료 (한국산업안전보건공단 산업안전보건연구원, 국가승인통계 제380002호) — 전국 만 15세 이상 취업자 50,195명 |
| **분석 대상** | 보건의료 직종(한국표준직업분류 241~246) 중 **교대근무자 전원** |
| **Target (종속변수 y)** | 🟧 `전신피로` — 지난 1년간 전신 피로 건강문제 있음(1) / 없음(0) (문96-F) |
| **문제 유형** | **분류(Classification)** — 타겟이 범주(0/1) |
| **모델** | 로지스틱 회귀, 결정트리, 랜덤포레스트 |
| **목적** | **의료기여 · 환자안전** (+ 공공: 보건의료 인력 노동안전) |

**배경**: 교대근무는 수면 부족과 피로로 이어지기 쉽고, 피로한 의료진은 투약 오류·알람 대응 지연 등 환자안전 사고 위험이 커집니다.

**선행연구**: 「한국 교대근무자의 피로에 대한 탐색적 조사 연구: 5차 근로환경조사를 이용하여」(한국웰니스학회지)는 교대근무자 4,774명을 분석해 **여성, 밤 근무·일요일 근무·하루 10시간 이상 근무 횟수가 많을수록, 기진맥진한 상태로 깨어날수록** 피로 위험이 높다고 보고했습니다. 본 과제는 최신 7차 자료로 이를 재확인하고, **보건의료 교대근무자**에 초점을 맞춰 예측 모델을 만듭니다.

### 연구 질문
1. 보건의료 종사자 중 교대근무자는 비교대근무자보다 전신 피로를 더 많이 겪는가?
2. 7차 자료에서도 선행연구(5차)의 피로 관련 요인이 재현되는가?
3. 근무 형태·수면 정보로 보건의료 교대근무자의 전신 피로를 예측할 수 있는가?

### 실행 방법
**런타임 → 모두 실행** → 업로드 창이 뜨면 `2023년_제7차_근로환경조사_원시자료.csv` 선택 (약 49MB, 업로드에 1~2분 소요)

## 1. 환경 설정

In [ ]:
try:
    import koreanize_matplotlib
except ImportError:
    !pip install -q koreanize-matplotlib
    import koreanize_matplotlib

import os, re, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

## 2. 원시자료 불러오기

근로환경조사 원시자료는 일반적인 CSV와 형식이 조금 다릅니다.
- 인코딩이 **cp949**(한글 윈도우 인코딩)
- 각 행이 큰따옴표로 감싸져 있고, 칸 구분은 **탭(\t)**
- 응답이 없는 칸은 공백(' ')

그래서 `pd.read_csv`를 그대로 쓰면 오류가 나므로, 한 줄씩 읽어 직접 나누는 함수를 만듭니다.

In [ ]:
def find_raw_file():
    hits = [f for f in glob.glob('*.csv') if '원시자료' in f]
    if hits:
        return hits[0]
    from google.colab import files
    print('근로환경조사 원시자료 CSV를 선택하세요')
    up = files.upload()
    return list(up.keys())[0]

def load_kwcs(path):
    text = open(path, 'rb').read().decode('cp949', errors='replace')
    rows = []
    for line in text.splitlines():
        if not line.strip():
            continue
        line = re.sub(r'",*\s*$', '', line)      # 줄 끝의 ",,, 제거
        if line.startswith('"'):
            line = line[1:]                        # 줄 앞의 " 제거
        rows.append(line.split('\t'))
    header, data = rows[0], rows[1:]
    data = [r + [''] * (len(header) - len(r)) for r in data]   # 칸 수가 모자란 행 보정
    df = pd.DataFrame(data, columns=header)
    df = df.replace(r'^\s*$', np.nan, regex=True)
    for c in df.columns:                           # 숫자로 바꿀 수 있는 열은 숫자로
        conv = pd.to_numeric(df[c], errors='coerce')
        if conv.notna().sum() >= df[c].notna().sum() * 0.99:
            df[c] = conv
    return df

FILE = find_raw_file()
kwcs = load_kwcs(FILE)
print('전체 응답자:', kwcs.shape[0], '명 /', kwcs.shape[1], '개 변수')   # 코드북 기준 50,195명

## 3. 분석 대상 추출

| 단계 | 조건 | 사용 변수 (코드북) |
|---|---|---|
| ① 보건의료 직종 | 한국표준직업분류 소분류 241 의료진료 전문가, 242 약사·한약사, **243 간호사**, 244 영양사, 245 치료·재활사 및 의료기사, **246 보건의료 관련 종사자** | `occ2` |
| ② 교대근무자 | 문81-E "교대 근무를 한다" = ① 그렇다 | `wtime_length5` |
| ③ Target 응답 있음 | 문96-F 전신 피로 = 있다/없다 (모름·거절 제외) | `heal_prob6` |

In [ ]:
OCC_NAMES = {241: '의료진료전문가', 242: '약사·한약사', 243: '간호사', 244: '영양사',
             245: '치료·재활사/의료기사', 246: '보건의료관련종사자'}

health = kwcs[kwcs['occ2'].between(241, 246)].copy()
health_shift = health[health['wtime_length5'] == 1].copy()
health_shift = health_shift[health_shift['heal_prob6'].isin([1, 2])]

print(f'전체 응답자            : {len(kwcs):,}명')
print(f'① 보건의료 직종        : {len(health):,}명')
print(f'② 그중 교대근무자      : {(health["wtime_length5"] == 1).sum():,}명')
print(f'③ Target 응답 있음 → 최종 분석 대상: {len(health_shift):,}명')
print()
print(health_shift['occ2'].map(OCC_NAMES).value_counts())

## 4. 변수 만들기 (코드북 기준 재코딩) + 비식별화

설문 응답 코드는 숫자지만 **의미의 방향이 제각각**입니다. 예를 들어 수면 문항은 1=매일, 5=전혀 없음이라 숫자가 클수록 *덜* 자주입니다. 분석하기 쉽도록 **"클수록 더 많이/더 자주"** 방향으로 바꾸고, 8(모름)·9(거절)은 결측으로 처리합니다.

| 새 변수 | 원 변수 | 문항 | 재코딩 |
|---|---|---|---|
| 여성 | gender | 성별 | 여=1, 남=0 |
| 나이 | age | 만 나이 | 그대로 |
| 순환교대 | wshift | 문81-1 교대 형태 | 순환교대=1, 그 외=0 |
| 밤근무일수 | wtime_night_a / wtime_night | 문56-A 한 달 밤근무(22~05시 2시간 이상) 일수 | 안 함=0, 모름·거절=결측 |
| 일요일근무일수 | wtime_sun_a / wtime_sun | 문56-B 한 달 일요일 근무 일수 | 〃 |
| 장시간근무일수 | wtime_long_a / wtime_long | 문56-D 한 달 하루 10시간 초과 근무 일수 | 〃 |
| 퀵리턴 | wtime_resilience | 문59 지난달 퇴근~출근 11시간 미만 있었음 | 예=1, 아니오=0 |
| 주당근무시간 | wtime_r | 실제 주당 근무시간(계산값) | 그대로 |
| 잠들기어려움 / 자주깸 / 기상시피로 | sleep1 / sleep2 / sleep3 | 문97 A·B·C 수면 문제 빈도 | 전혀없음=0 … 매일=4 |
| 업무스트레스 | wsituation12 | 문48-L 업무에서 스트레스를 받는다 | 전혀=0 … 항상=4 |
| 화난환자응대 | hazard_psy2 | 문66-H 화난 고객·환자를 다루는 시간 | 전혀없음=0 … 근무시간 내내=6 |
| 🟧 **전신피로** | heal_prob6 | 문96-F 지난 1년 전신 피로 | 있다=1, 없다=0 |

**비식별화**: 원시자료는 이미 국가통계로 비식별 처리되어 공개된 자료입니다. 추가로 조사 ID(`id`)는 순번(P001…)으로 바꾸고, 가중치·가구 정보 등 분석에 불필요한 변수는 모두 제외합니다.

In [ ]:
FREQ5 = {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}       # 매일 … 전혀없음 → 4 … 0

def month_days(d, flag, days):
    out = pd.Series(np.nan, index=d.index)
    out[d[flag] == 0] = 0
    yes = d[flag] == 1
    out[yes] = d.loc[yes, days]
    return out

def make_features(d):
    f = pd.DataFrame(index=d.index)
    f['직종'] = d['occ2'].map(OCC_NAMES)
    f['여성'] = (d['gender'] == 2).astype(int)
    f['나이'] = d['age']
    f['순환교대'] = (d['wshift'] == 3).astype(int)
    f['밤근무일수'] = month_days(d, 'wtime_night_a', 'wtime_night')
    f['일요일근무일수'] = month_days(d, 'wtime_sun_a', 'wtime_sun')
    f['장시간근무일수'] = month_days(d, 'wtime_long_a', 'wtime_long')
    f['퀵리턴'] = d['wtime_resilience'].map({1: 1, 2: 0})
    f['주당근무시간'] = d['wtime_r']
    f['잠들기어려움'] = d['sleep1'].map(FREQ5)
    f['자주깸'] = d['sleep2'].map(FREQ5)
    f['기상시피로'] = d['sleep3'].map(FREQ5)
    f['업무스트레스'] = d['wsituation12'].map(FREQ5)
    f['화난환자응대'] = d['hazard_psy2'].map({1: 6, 2: 5, 3: 4, 4: 3, 5: 2, 6: 1, 7: 0})
    f['전신피로'] = d['heal_prob6'].map({1: 1, 2: 0})
    # ── Target 측정 근거 확인용 변수 (모델 입력에는 사용하지 않음) ──
    f['전신피로_업무관련'] = d['heal_prob6_1'].map({1: 1, 2: 0})
    f['결근일수'] = d['heal_abs1'].where(d['heal_abs1'] < 888)
    f['아파도출근'] = d['heal_wsick1'].map({1: 1, 2: 0, 7: 0})
    f['주관적건강나쁨'] = d['heal_cond'].map({1: 0, 2: 0, 3: 0, 4: 1, 5: 1})
    who = d[['who1', 'who2', 'who3', 'who4', 'who5']].where(lambda x: x <= 6)
    f['WHO5웰빙점수'] = ((6 - who).sum(axis=1, min_count=5)) * 4          # 0~100, 높을수록 좋음
    f['퇴근시기진맥진'] = d['weng4'].map({1: 4, 2: 3, 3: 2, 4: 1, 5: 0})    # 0(전혀)~4(항상)
    return f

df = make_features(health_shift).reset_index(drop=True)
df.insert(0, 'ID', [f'P{i:03d}' for i in range(1, len(df) + 1)])
print(df.shape)
df.head()

## 5. Target과 독립·종속변수 정리 🟧

| 구분 | 독립변수 (x, 입력) | 종속변수 (y, 결과) |
|---|---|---|
| **개념** | 결과를 설명하는 변수 | 설명·예측하려는 결과 |
| **이 과제에서** | 근무 요인(성별·나이·순환교대·밤근무·일요일근무·장시간근무·퀵리턴·주당근무시간) + 수면 요인(잠들기어려움·자주깸·기상시피로) | 🟧 **전신피로** (= Target) |
| **코드에서** | `X` | `y` |

→ 모델은 **y = f(x)** 의 f를 데이터에서 학습합니다.

**⚠️ 두 가지 주의점**
1. **'원인'이라고 단정하지 않는다** — 근로환경조사는 한 시점의 **관찰 자료(횡단면 조사)** 이므로 "관련이 있다 / 예측에 도움이 된다"까지만 말할 수 있습니다.
2. **'기상시피로'는 Target과 개념이 겹친다** — "자고 일어나도 지치고 피곤하다"는 전신 피로와 매우 비슷한 증상이라, 넣으면 성능이 오르지만 사실상 피로로 피로를 맞히는 셈일 수 있습니다. 그래서 **모델 A(근무 요인만)** 와 **모델 B(근무+수면)** 를 나눠 비교합니다.

In [ ]:
TARGET = '전신피로'
WORK = ['여성', '나이', '순환교대', '밤근무일수', '일요일근무일수', '장시간근무일수', '퀵리턴', '주당근무시간']
SLEEP = ['잠들기어려움', '자주깸', '기상시피로']

def highlight_target(col):
    return ['background-color: #FFA94D; font-weight: bold' if col.name == TARGET else '' for _ in col]

n1 = int(df[TARGET].sum())
print(f'분석 대상 {len(df)}명 중 전신피로 있음 {n1}명 ({n1/len(df):.1%}), 없음 {len(df)-n1}명')
print('▶ 독립변수 (x) — 모델 A:', WORK)
print('▶ 독립변수 (x) — 모델 B:', WORK + SLEEP)
print('▶ 종속변수 (y):', TARGET)
df.head(10).style.apply(highlight_target)

## 5-1. Target '전신피로'는 무엇으로, 어떻게 측정되었나?

### ① 측정 도구와 방법 (조사표·코드북·자료이용설명서 기준)

| 항목 | 내용 |
|---|---|
| **문항** | 문96. "**지난 1년 동안** 다음과 같은 건강상 문제가 있었습니까?" → **F. 전신 피로** |
| **응답 보기** | ① 있다 ② 없다 ⑧ 모름/무응답 ⑨ 거절 |
| **후속 문항** | 문96-1-F. (있다고 답한 경우) "귀하가 하시는 일 때문에 발생했습니까?" ① 그렇다 ② 아니다 |
| **변수명** | `heal_prob6` (후속: `heal_prob6_1`) |
| **조사 방식** | 전문면접원의 **가구 방문 1:1 면접조사**, 태블릿 PC 전자설문(TAPI) — 전체의 98.9%가 현장면접 |
| **조사 시기** | 2023년 (제7차 근로환경조사) |
| **전국 분포** | 전체 응답자 50,195명 중 '있다' 11,351명(22.6%) (코드북 빈도표) |
| **측정 성격** | **자기보고(self-report) 단일 문항, 이분형(있다/없다)** |

### ② 한계를 분명히 하면
- 근로환경조사에는 **혈액검사, 활동량계(액티그래피), 반응시간 검사 같은 생체·객관적 측정이 없습니다.** 전신 피로는 응답자 본인의 보고입니다.
- 표준화된 다문항 피로 척도(FSS, MFI 등)가 아닌 **단일 문항**입니다.

### ③ 그래서 '같은 조사 안의 다른 자료'로 타당성을 확인합니다 (수렴 타당도)
전신피로를 '있다'고 답한 사람이 정말 피로한 사람이라면, **다른 문항·행동 지표에서도 나쁜 상태**로 나타나야 합니다.

| 확인 지표 | 문항 | 성격 |
|---|---|---|
| **결근일수** | 문89. 지난 1년 건강 문제로 결근한 날 | **행동 지표**(일수) — 자기보고 중 가장 객관적 |
| **아파도 출근(프리젠티즘)** | 문90. 업무에 큰 지장이 있을 정도로 아픈데도 일함 | 행동 지표 |
| **주관적 건강 나쁨** | 문93. 건강 상태 '나쁜 편/매우 나쁨' | 전반적 건강 |
| **WHO-5 웰빙 점수** | 문98 A~E (WHO-5 웰빙지수, 0~100점, 50점 이하=웰빙 저하) | **국제 표준화 척도** |
| **퇴근 시 기진맥진** | 문73-D. 퇴근할 때 신체적으로 기진맥진 | 피로 관련 다른 문항 |

In [ ]:
VALID = ['결근일수', '아파도출근', '주관적건강나쁨', 'WHO5웰빙점수', '퇴근시기진맥진']

def validity_table(data, label):
    rows = []
    for col in VALID:
        g0 = data.loc[data[TARGET] == 0, col].dropna()
        g1 = data.loc[data[TARGET] == 1, col].dropna()
        if set(data[col].dropna().unique()) <= {0, 1}:                 # 이분형 → 비율, 카이제곱
            tab = pd.crosstab(data[TARGET], data[col])
            pval = stats.chi2_contingency(tab)[1]
            v0, v1, unit = f'{g0.mean():.1%}', f'{g1.mean():.1%}', '비율'
        else:                                                          # 연속형 → 평균, Mann-Whitney
            pval = stats.mannwhitneyu(g0, g1).pvalue
            v0, v1, unit = f'{g0.mean():.2f}', f'{g1.mean():.2f}', '평균'
        rows.append({'확인 지표': col, '단위': unit, '전신피로 없음': v0, '전신피로 있음': v1,
                     'p값': round(pval, 4), '예상 방향과 일치': '✔' if pval < .05 else '(유의하지 않음)'})
    print(f'[{label}]')
    return pd.DataFrame(rows).set_index('확인 지표')

valid_health = validity_table(df, f'보건의료 교대근무자 n = {len(df)}')
display(valid_health)

all_workers = make_features(kwcs)
all_workers = all_workers[all_workers[TARGET].notna()]
valid_all = validity_table(all_workers, f'참고: 전체 취업자 n = {len(all_workers):,}')
display(valid_all)

work_related = df['전신피로_업무관련'].dropna()
print(f'보건의료 교대근무자 중 전신피로 있음 {int(df[TARGET].sum())}명 → '
      f'일 때문에 발생했다고 응답 {int(work_related.sum())}명 ({work_related.mean():.1%})')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.8))
for ax, col in zip(axes, VALID):
    m = df.groupby(TARGET)[col].mean()
    is_rate = set(df[col].dropna().unique()) <= {0, 1}
    vals = m.values * (100 if is_rate else 1)
    ax.bar(['없음', '있음'], vals, color=['tab:blue', 'tab:orange'])
    for i, v in enumerate(vals):
        ax.text(i, v, f'{v:.1f}{"%" if is_rate else ""}', ha='center', va='bottom')
    ax.set_title(col + (' (%)' if is_rate else ' (평균)'))
    ax.set_xlabel('전신피로')
fig.suptitle('Target 타당성 확인: 전신피로 응답자는 다른 지표에서도 나쁜가? (보건의료 교대근무자)', fontsize=13)
plt.tight_layout(); plt.show()

**해석 방법**: ✔가 많을수록 '전신피로 있음' 응답이 실제 건강·기능 저하와 함께 나타난다는 뜻이므로, 단일 자기보고 문항이지만 Target으로 쓸 근거가 됩니다. 보건의료 교대근무자(202명)는 표본이 작아 유의하지 않은 지표가 있을 수 있어, **전체 취업자 결과를 함께 제시**합니다.

> 📌 **진짜 객관적 피로 측정이 필요하다면?** 활동량계(수면시간·수면효율), 정신운동각성검사(PVT) 반응시간, 타액 코르티솔 등이 쓰입니다. 예를 들어 PhysioNet의 24시간 수면박탈 연구는 CANTAB 반응시간 검사로 수면박탈 후 반응속도 저하를 객관적으로 측정했습니다. 본 과제 자료에는 이런 측정이 없으므로 **향후 과제**로 제시합니다.

## 6. 탐색적 분석 (EDA)

### 6-1. 보건의료 종사자: 교대근무자 vs 비교대근무자의 전신 피로율

In [ ]:
h_all = health[health['heal_prob6'].isin([1, 2]) & health['wtime_length5'].isin([1, 2])].copy()
h_all['교대여부'] = h_all['wtime_length5'].map({1: '교대근무', 2: '비교대근무'})
h_all['전신피로'] = h_all['heal_prob6'].map({1: 1, 2: 0})

tab = pd.crosstab(h_all['교대여부'], h_all['전신피로'])
chi2, p_chi, _, _ = stats.chi2_contingency(tab)
rate = h_all.groupby('교대여부')['전신피로'].agg(['size', 'mean'])
rate.columns = ['인원', '전신피로율']
display(rate.style.format({'전신피로율': '{:.1%}'}))
print(f'카이제곱 검정: χ² = {chi2:.2f}, p = {p_chi:.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(rate.index, rate['전신피로율'] * 100, color=['tab:orange', 'tab:blue'])
for i, v in enumerate(rate['전신피로율'] * 100):
    ax.text(i, v + 0.8, f'{v:.1f}%', ha='center', fontsize=12)
ax.set_ylabel('지난 1년 전신 피로 경험 (%)')
ax.set_title('보건의료 종사자의 교대근무 여부별 전신 피로율')
ax.set_ylim(0, 40)
plt.tight_layout(); plt.show()

In [ ]:
comp = df.groupby(TARGET)[WORK + SLEEP].mean().T.round(2)
comp.columns = ['전신피로 없음(0)', '전신피로 있음(1)']
comp['차이(1-0)'] = (comp.iloc[:, 1] - comp.iloc[:, 0]).round(2)
comp

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
show = ['나이', '밤근무일수', '일요일근무일수', '주당근무시간', '잠들기어려움', '자주깸', '기상시피로', '장시간근무일수']
for ax, col in zip(axes.flat, show):
    data = [df.loc[df[TARGET] == t, col].dropna() for t in (0, 1)]
    ax.boxplot(data, labels=['없음', '있음'], widths=.5, patch_artist=True,
               boxprops=dict(facecolor='lightgrey'))
    for t, d in enumerate(data, 1):
        ax.scatter(np.random.normal(t, .06, len(d)), d, s=10, alpha=.4,
                   color='tab:orange' if t == 2 else 'tab:blue')
    ax.set_title(col); ax.set_xlabel('전신피로')
fig.suptitle('보건의료 교대근무자: 전신피로 유무에 따른 근무·수면 특성', fontsize=14)
plt.tight_layout(); plt.show()

### 6-2. 근무·수면 요인과 전신피로의 상관관계 히트맵

앞에서 만든 `df`, `WORK`, `SLEEP`, `TARGET`을 그대로 사용합니다. 모델 B의 입력 변수 11개와 전신피로를 표시합니다.

- 순서형 수면 점수 등을 고려하여 **Spearman 순위상관계수**를 사용합니다.
- 빨강은 양의 관계, 파랑은 음의 관계입니다. 계수는 −1~1이며, 0에 가까울수록 단조 관계가 약합니다.
- 결측값을 채우지 않고 각 변수 쌍에 모두 응답한 사람만 계산합니다. `heatmap_n`에 칸별 분석 인원이 저장됩니다.
- ID·직종 및 피로 응답자에게만 묻는 후속 문항은 포함하지 않습니다.
- 조사 가중치·교란요인을 보정하지 않은 탐색 결과입니다. 인과관계나 통계적 유의성을 의미하지 않습니다. 특히 기상시피로는 전신피로와 개념이 겹칩니다.


In [ ]:
import seaborn as sns

heatmap_cols = WORK + SLEEP + [TARGET]
heatmap_data = df[heatmap_cols].apply(pd.to_numeric, errors='raise')
heatmap_corr = heatmap_data.corr(method='spearman', min_periods=3)
heatmap_valid = heatmap_data.notna().astype(int)
heatmap_n = heatmap_valid.T @ heatmap_valid

fig, ax = plt.subplots(figsize=(13, 10))
heatmap_mask = np.triu(np.ones_like(heatmap_corr, dtype=bool), k=1)
sns.heatmap(heatmap_corr, mask=heatmap_mask, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True,
            linewidths=.6, annot_kws={'size': 9},
            cbar_kws={'label': 'Spearman 상관계수', 'shrink': .75}, ax=ax)
ax.set_title(f'보건의료 교대근무자: 근무·수면 요인과 전신피로\n총 {len(df)}명 · 변수 쌍별 결측 제외', fontsize=15, pad=20)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
plt.setp(ax.get_yticklabels(), rotation=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    if label.get_text() == TARGET:
        label.set_color('#B45309')
        label.set_fontweight('bold')
fig.tight_layout()
plt.show()


### 6-3. 전신피로와의 상관관계만 보기

상관계수의 절댓값이 큰 순서입니다. 각 행의 n은 실제 계산에 사용한 인원입니다. 이 순위는 예측모델의 변수 중요도나 원인 순위가 아닙니다.

In [ ]:
heatmap_target = heatmap_corr.loc[WORK + SLEEP, TARGET]
heatmap_order = heatmap_target.abs().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(6.5, 7.5))
sns.heatmap(heatmap_target.loc[heatmap_order].to_frame(), annot=True,
            fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=1, cbar_kws={'label': 'Spearman 상관계수'}, ax=ax)
ax.set_yticklabels([f'{col} (n={heatmap_n.loc[col, TARGET]})' for col in heatmap_order], rotation=0)
ax.set_xticklabels(['전신피로 (있음=1)'], rotation=0)
ax.set_title('전신피로와의 상관관계', fontsize=15, pad=18)
ax.set_ylabel('')
fig.tight_layout()
plt.show()

display(pd.DataFrame({
    'Spearman 상관계수': heatmap_target.loc[heatmap_order],
    '유효 표본 수': heatmap_n.loc[heatmap_order, TARGET]
}).round(3))


## 7. 통계 분석 — 로지스틱 회귀 (오즈비)

**오즈비(OR)** 해석: OR > 1이면 그 변수가 1 증가할 때 전신 피로 가능성이 커지는 방향, p < 0.05면 통계적으로 유의.

### 7-1. 선행연구 재현: 7차 전체 교대근무자(전 직종)

In [ ]:
import statsmodels.api as sm

def logit_or(data, cols):
    d = data[cols + [TARGET]].dropna(subset=[TARGET])
    X = sm.add_constant(d[cols].fillna(d[cols].median()))
    res = sm.Logit(d[TARGET], X).fit(disp=0)
    ci = np.exp(res.conf_int())
    out = pd.DataFrame({'OR': np.exp(res.params), '95%CI 하한': ci[0], '95%CI 상한': ci[1],
                        'p값': res.pvalues}).drop('const')
    out['유의(p<.05)'] = np.where(out['p값'] < .05, '✔', '')
    return out.round(3), len(d)

all_shift = kwcs[kwcs['wtime_length5'] == 1]
all_df = make_features(all_shift)
or_all, n_all = logit_or(all_df, WORK)
print(f'7차 전체 교대근무자 n = {n_all:,}명 (근무 요인만)')
or_all

In [ ]:
prior = pd.DataFrame({
    '5차 선행연구 결과': ['피로 위험 ↑', '(초록에 언급 없음)', '(초록에 언급 없음)', '피로 위험 ↑', '피로 위험 ↑', '피로 위험 ↑', '(초록에 언급 없음)', '(초록에 언급 없음)'],
}, index=WORK)
prior['7차 재분석(본 과제)'] = [
    ('유의 ↑' if (r['p값'] < .05 and r['OR'] > 1) else '유의 ↓' if (r['p값'] < .05) else '유의하지 않음')
    for _, r in or_all.iterrows()]
prior

### 7-2. 보건의료 교대근무자 (모델 B 변수 전체)

In [ ]:
or_health, n_h = logit_or(df, WORK + SLEEP)
print(f'보건의료 교대근무자 n = {n_h}명')
or_health

## 8. 머신러닝 분류 모델

- **모델 A**: 근무 요인만 (근무표만 보고 예측 가능한지)
- **모델 B**: 근무 + 수면 요인
- 알고리즘: 로지스틱 회귀 / 결정트리(깊이 3) / 랜덤포레스트
- **검증**: 5-fold 층화 교차검증 × 10회 반복 (표본이 202명으로 작아 한 번 나눈 결과는 운에 좌우되므로 50번 평가한 평균 사용)
- **클래스 불균형**: 전신피로가 약 30%라 `class_weight='balanced'`로 소수 클래스에 가중치
- **평가 지표**
  - **AUC**: 0.5 = 동전 던지기, 1.0 = 완벽. 0.7 이상이면 쓸 만한 수준으로 봄
  - **민감도(Recall)**: 실제 피로자를 놓치지 않는 비율 — 환자안전 목적상 가장 중요
  - **균형정확도**: 두 그룹 정확도의 평균 (불균형 데이터에서 일반 정확도보다 공정)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

MODELS = {
    '로지스틱 회귀': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                  LogisticRegression(class_weight='balanced', max_iter=1000)),
    '결정트리':     make_pipeline(SimpleImputer(strategy='median'),
                                  DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)),
    '랜덤포레스트': make_pipeline(SimpleImputer(strategy='median'),
                                  RandomForestClassifier(n_estimators=300, max_depth=4,
                                                         class_weight='balanced', random_state=42)),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
y = df[TARGET]

rows = []
for set_name, cols in [('A: 근무 요인', WORK), ('B: 근무+수면', WORK + SLEEP)]:
    for m_name, model in MODELS.items():
        r = cross_validate(model, df[cols], y, cv=cv,
                           scoring=['roc_auc', 'recall', 'balanced_accuracy'])
        rows.append({'변수 세트': set_name, '모델': m_name,
                     'AUC': r['test_roc_auc'].mean(),
                     '민감도': r['test_recall'].mean(),
                     '균형정확도': r['test_balanced_accuracy'].mean()})
perf = pd.DataFrame(rows).set_index(['변수 세트', '모델']).round(3)
perf

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
auc_tab = perf['AUC'].unstack(0)
auc_tab.plot(kind='bar', ax=ax, color=['lightgrey', 'tab:orange'], rot=0)
ax.axhline(.5, color='k', ls='--', lw=1); ax.text(2.35, .505, '우연(0.5)', fontsize=9)
ax.axhline(.7, color='tab:green', ls=':', lw=1); ax.text(2.35, .705, '쓸 만한 수준(0.7)', fontsize=9)
ax.set_ylim(.3, .8); ax.set_ylabel('AUC (교차검증 평균)')
ax.set_title('모델 A(근무) vs 모델 B(근무+수면) 예측 성능')
plt.tight_layout(); plt.show()

### 8-1. 민감도 분석 — Target을 다른 피로 문항으로 바꿔도 결과가 같을까?
단일 문항 Target의 약점을 보완하기 위해, **'퇴근 시 기진맥진'을 항상/대부분 그렇다(3~4점)로 답한 경우**를 대체 Target으로 놓고 같은 모델을 다시 돌립니다. 결론의 방향(근무 요인만으로는 예측이 어렵고, 수면 정보가 도움이 됨)이 같으면 결과가 Target 정의에 크게 좌우되지 않는다는 근거가 됩니다.

In [ ]:
alt = df.dropna(subset=['퇴근시기진맥진']).copy()
alt['대체Target_기진맥진'] = (alt['퇴근시기진맥진'] >= 3).astype(int)
print(f"대체 Target 양성: {alt['대체Target_기진맥진'].sum()}명 / {len(alt)}명 ({alt['대체Target_기진맥진'].mean():.1%})")
print(f"원 Target(전신피로)과의 일치도: {(alt['대체Target_기진맥진'] == alt[TARGET]).mean():.1%}")

rows = []
for set_name, cols in [('A: 근무 요인', WORK), ('B: 근무+수면', WORK + SLEEP)]:
    r = cross_validate(MODELS['로지스틱 회귀'], alt[cols], alt['대체Target_기진맥진'], cv=cv,
                       scoring=['roc_auc', 'recall', 'balanced_accuracy'])
    rows.append({'변수 세트': set_name, 'AUC': r['test_roc_auc'].mean(),
                 '민감도': r['test_recall'].mean(), '균형정확도': r['test_balanced_accuracy'].mean()})
sens = pd.DataFrame(rows).set_index('변수 세트').round(3)
sens

In [ ]:
tree = make_pipeline(SimpleImputer(strategy='median'),
                     DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42))
tree.fit(df[WORK + SLEEP], y)
plt.figure(figsize=(16, 7))
plot_tree(tree[-1], feature_names=WORK + SLEEP, class_names=['피로 없음', '피로 있음'],
          filled=True, rounded=True, fontsize=9, proportion=True)
plt.title('결정트리 (모델 B) — 전신 피로 판별 규칙'); plt.show()

rf = MODELS['랜덤포레스트'].fit(df[WORK + SLEEP], y)
imp = pd.Series(rf[-1].feature_importances_, index=WORK + SLEEP).sort_values()
imp.plot(kind='barh', figsize=(6, 4), color='tab:orange', title='랜덤포레스트 변수 중요도 (모델 B)')
plt.tight_layout(); plt.show()

## 9. 활용 방안

| 구분 | 활용 |
|---|---|
| **의료기여 · 환자안전** (주 목적) | 교대근무 간호사·보건의료 인력 대상 **수면 셀프체크**(특히 "자고 일어나도 피곤한가") → 고위험자 조기 발견, 근무 조정·투약 이중확인 강화 |
| **공공 (노동안전)** | 퀵리턴(휴식 11시간 미만)·밤근무·일요일근무 등 **근무 편성 기준 개선의 근거** — 전 직종 교대근무자에서 확인된 위험 요인 |
| **병원 운영** | 근무표 데이터 + 정기 수면 설문을 결합한 피로 모니터링 대시보드 |

**윤리적 고려**: 결과는 인사평가·불이익이 아니라 근무 조정과 지원에만 사용해야 합니다.

## 10. 결론

In [ ]:
r_rate = rate['전신피로율']
auc = perf['AUC']
sig_all = or_all[or_all['p값'] < .05]
sig_up = [f'{k}(OR {v:.2f})' for k, v in sig_all['OR'].items() if v > 1]
b_best = auc.loc['B: 근무+수면'].idxmax()

print('■ 1. 교대근무와 전신 피로')
print(f'   보건의료 종사자 중 교대근무자 전신피로율 {r_rate["교대근무"]:.1%} vs 비교대 {r_rate["비교대근무"]:.1%} (χ² p = {p_chi:.4f})')
print('■ 2. 선행연구 재현 (7차 전체 교대근무자 n = %d)' % n_all)
print('   유의하게 피로 위험을 높인 근무 요인:', ', '.join(sig_up))
print('■ 3. 보건의료 교대근무자 예측 모델 (n = %d)' % len(df))
print(f'   모델 A(근무 요인만) AUC {auc.loc["A: 근무 요인"].min():.2f}~{auc.loc["A: 근무 요인"].max():.2f}')
print(f'   모델 B(근무+수면)   AUC {auc.loc["B: 근무+수면"].min():.2f}~{auc.loc["B: 근무+수면"].max():.2f} (최고: {b_best})')
print(f'   보건의료 교대근무자에서 유의한 변수: {", ".join(or_health[or_health["p값"] < .05].index)}')
print('■ 4. Target 타당성 (전신피로 응답자가 다른 지표에서도 나쁜가)')
print('   보건의료 교대근무자: 유의한 지표', ', '.join(valid_health[valid_health['p값'] < .05].index) or '없음')
print('   전체 취업자:        유의한 지표', ', '.join(valid_all[valid_all['p값'] < .05].index) or '없음')
print(f'■ 5. 민감도 분석 (대체 Target=퇴근 시 기진맥진): 모델 A AUC {sens.loc["A: 근무 요인","AUC"]:.2f}, 모델 B AUC {sens.loc["B: 근무+수면","AUC"]:.2f}')

### 결론 요약

**한 줄 결론: 보건의료 종사자 중 교대근무자는 전신 피로를 더 많이 겪으며(약 30% vs 17%), 전 직종 교대근무자에서는 퀵리턴·밤근무·일요일근무 등 근무 편성 요인이 피로와 관련이 있었다. 그러나 보건의료 교대근무자만 놓고 보면 근무 요인만으로는 피로를 예측하기 어려웠고, '자고 일어나도 피곤한' 수면 상태를 함께 보아야 예측력이 생겼다.**

1. **교대근무 자체가 위험 신호** — 같은 보건의료 직종 안에서도 교대근무자의 전신피로율이 비교대근무자보다 뚜렷하게 높았다.
2. **선행연구(5차)는 대부분 재현됨** — 7차 전체 교대근무자에서 여성, 밤근무 일수, 일요일근무 일수가 많을수록 피로 위험이 높았다. 추가로 **퀵리턴(휴식 11시간 미만)** 이 가장 큰 오즈비를 보였다. 다만 하루 10시간 초과 근무는 이번 자료에서 유의하지 않았다.
3. **보건의료 교대근무자 안에서는 근무 요인만으로 예측 불가(AUC≈0.5)** — 대부분이 여성·순환교대·비슷한 근무표로 일해 근무 요인의 차이 자체가 작기 때문으로 보인다.
4. **수면 정보가 핵심** — 수면 요인을 넣으면 AUC가 0.56~0.61로 올랐고, '기상시피로'가 가장 중요한 변수였다. 단, 이 문항은 Target과 개념이 겹쳐 실제 예측력보다 과대평가되었을 수 있고, AUC 0.6대는 **선별 도구로 바로 쓰기엔 부족한 수준**이다.
5. **Target 측정의 타당성** — 전신피로는 면접조사의 자기보고 단일 문항이지만, 전체 취업자 5만여 명에서 전신피로 응답자는 결근일수·아파도 출근·주관적 건강 나쁨·WHO-5 웰빙 저하·퇴근 시 기진맥진 **5개 지표 모두에서 유의하게 나빴다.** 보건의료 교대근무자(202명)에서는 표본이 작아 '퇴근 시 기진맥진'만 유의했다. 또 전신피로 응답자의 85%가 '일 때문에 발생했다'고 답했다.
6. **민감도 분석** — Target을 '퇴근 시 기진맥진'으로 바꿔도 근무 요인만으로는 예측이 안 되고(AUC 0.45), 수면을 더하면 나아지는(AUC 0.57) 같은 패턴이 나와, 결론이 Target 정의에 크게 좌우되지 않았다.
7. **시사점** — 병원 현장에서는 근무표만으로 위험군을 가려내기보다, **근무 편성 기준(퀵리턴 금지, 연속 야간 제한)을 지키고, 정기적인 수면·피로 셀프체크를 함께 운영**하는 것이 합리적이다.

## 11. 한계점
- **횡단면 관찰 자료** → 인과관계가 아닌 관련성만 확인
- **보건의료 교대근무자 202명**으로 표본이 작고, 대부분 여성·간호사라 다른 직종에 일반화 어려움
- **Target이 단일 문항 자기보고**("지난 1년 전신 피로 있음")로, 표준화된 피로 척도(FSS 등)나 **객관적 생체 측정(활동량계, 반응시간 검사 등)이 아님** → 같은 조사 내 다른 지표로 타당성을 확인했으나, 보건의료 교대근무자 안에서는 일부 지표만 유의
- '기상시피로'와 Target의 **개념 중복** 가능성
- 조사 **가중치(wt2)를 적용하지 않은** 표본 기준 분석 → 전국 모집단 추정치가 아님
- 근무표의 세부 정보(연속 야간 일수, 교대 방향 등)는 조사에 없음

## 12. 조별 폴더 제출용 엑셀 (타겟 색 표시)
시트: ① 데이터(분석 대상 전체, 타겟 열 주황색) ② 변수설명(타겟 행 주황색) ③ Target·모델·활용·결론 ④ 선행연구 비교 ⑤ Target 측정 근거

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

ORANGE, LIGHT = PatternFill('solid', start_color='FFA94D'), PatternFill('solid', start_color='FFE8CC')
HEAD = PatternFill('solid', start_color='D9E2F3')
BOLD, NORMAL = Font(name='Arial', bold=True), Font(name='Arial')
thin = Side(style='thin', color='BFBFBF'); BOX = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_table(ws, frame, start_row=1, target_col=None):
    for j, c in enumerate(frame.columns, 1):
        cell = ws.cell(row=start_row, column=j, value=str(c))
        cell.font, cell.border, cell.alignment = BOLD, BOX, Alignment(horizontal='center', wrap_text=True)
        cell.fill = ORANGE if c == target_col else HEAD
        ws.column_dimensions[get_column_letter(j)].width = max(10, len(str(c)) * 1.8)
    for i, row in enumerate(frame.itertuples(index=False), start_row + 1):
        for j, v in enumerate(row, 1):
            v = v.item() if hasattr(v, 'item') else v
            cell = ws.cell(row=i, column=j, value=None if pd.isna(v) else v)
            cell.font, cell.border, cell.alignment = NORMAL, BOX, Alignment(horizontal='center', wrap_text=True)
            if frame.columns[j - 1] == target_col:
                cell.fill, cell.font = LIGHT, BOLD

wb = Workbook()
ws = wb.active; ws.title = '데이터'
ws['A1'] = f'보건의료 교대근무자 {len(df)}명 — 제7차(2023) 근로환경조사 원시자료 추출 (주황색 = Target)'
ws['A1'].font = Font(name='Arial', bold=True, size=12)
write_table(ws, df, start_row=3, target_col=TARGET)
ws.freeze_panes = 'B4'

ws2 = wb.create_sheet('변수설명')
var_desc = pd.DataFrame([
    ('ID', '-', '순번 식별자(원 조사 ID 대체)', '분석 제외'),
    ('직종', 'occ2', '직업 소분류 241~246', '참고'),
    ('여성', 'gender', '여=1, 남=0', '독립변수(x)'),
    ('나이', 'age', '만 나이', '독립변수(x)'),
    ('순환교대', 'wshift', '문81-1 순환교대=1', '독립변수(x)'),
    ('밤근무일수', 'wtime_night_a/_night', '문56-A 한 달 밤근무 일수', '독립변수(x)'),
    ('일요일근무일수', 'wtime_sun_a/_sun', '문56-B 한 달 일요일 근무 일수', '독립변수(x)'),
    ('장시간근무일수', 'wtime_long_a/_long', '문56-D 한 달 10시간 초과 근무 일수', '독립변수(x)'),
    ('퀵리턴', 'wtime_resilience', '문59 휴식 11시간 미만 있음=1', '독립변수(x)'),
    ('주당근무시간', 'wtime_r', '실제 주당 근무시간', '독립변수(x)'),
    ('잠들기어려움', 'sleep1', '문97-A 0(전혀)~4(매일)', '독립변수(x), 모델 B'),
    ('자주깸', 'sleep2', '문97-B 0~4', '독립변수(x), 모델 B'),
    ('기상시피로', 'sleep3', '문97-C 0~4 (Target과 개념 중복 주의)', '독립변수(x), 모델 B'),
    ('업무스트레스', 'wsituation12', '문48-L 0(전혀)~4(항상)', '참고'),
    ('화난환자응대', 'hazard_psy2', '문66-H 0(전혀)~6(근무시간 내내)', '참고'),
    ('전신피로', 'heal_prob6', '문96-F 지난 1년 전신 피로 있음=1', '★ 종속변수(y) = TARGET'),
], columns=['변수명', '원 변수(코드북)', '설명', '역할'])
write_table(ws2, var_desc)
for i, v in enumerate(var_desc['변수명'], 2):
    if v == TARGET:
        for j in range(1, 5): ws2.cell(row=i, column=j).fill = ORANGE
for col, w in zip('ABCD', [16, 22, 40, 24]): ws2.column_dimensions[col].width = w

ws3 = wb.create_sheet('Target_모델_활용')
best_b = perf.loc['B: 근무+수면']
summary = pd.DataFrame([
    ('주제', '보건의료 교대근무자의 전신 피로 예측'),
    ('데이터', f'제7차(2023) 근로환경조사 원시자료 50,195명 중 보건의료 직종 교대근무자 {len(df)}명 전원'),
    ('Target', '전신피로 (문96-F 지난 1년 전신 피로 있음=1)'),
    ('문제 유형', '분류 (타겟 0/1)'),
    ('독립변수(x)', '모델 A: ' + ', '.join(WORK) + ' / 모델 B: + ' + ', '.join(SLEEP)),
    ('종속변수(y)', '전신피로 = Target'),
    ('모델', '로지스틱 회귀, 결정트리(깊이3), 랜덤포레스트 — 5-fold 층화 교차검증 ×10회'),
    ('성능', ' / '.join(f'{s} {m}: AUC {r.AUC:.2f}' for (s, m), r in perf.iterrows())),
    ('목적', '의료기여·환자안전 (+ 공공: 보건의료 인력 노동안전)'),
    ('활용', '수면·피로 셀프체크로 고위험자 조기 발견, 퀵리턴·밤근무 등 근무 편성 기준 개선 근거'),
    ('결론', f'보건의료 교대근무자 전신피로율 {r_rate["교대근무"]:.1%}(비교대 {r_rate["비교대근무"]:.1%}). '
             '전 직종 교대근무자에서 퀵리턴·밤근무·일요일근무 등이 피로와 관련. '
             '보건의료 교대근무자에서는 근무 요인만으로 예측 어려웠고 수면 정보를 더하면 예측력 향상(단, 선별 도구로는 부족).'),
    ('Target 측정', '면접조사 자기보고 단일문항(문96-F). 전체 취업자에서 결근·프리젠티즘·주관적건강·WHO-5·기진맥진과 모두 유의하게 연관(타당성 근거). 객관적 생체측정은 없음'),
    ('한계', '횡단면 관찰자료, 표본 202명, 단일문항 자기보고(객관적 측정 없음), 기상시피로-Target 개념 중복, 가중치 미적용'),
], columns=['항목', '내용'])
write_table(ws3, summary)
for i, k in enumerate(summary['항목'], 2):
    ws3.cell(row=i, column=2).alignment = Alignment(wrap_text=True, vertical='top')
    if k == 'Target':
        ws3.cell(row=i, column=1).fill = ORANGE; ws3.cell(row=i, column=2).fill = LIGHT
ws3.column_dimensions['A'].width = 14; ws3.column_dimensions['B'].width = 100

ws4 = wb.create_sheet('선행연구비교')
write_table(ws4, or_all.reset_index().rename(columns={'index': '변수'}).merge(
    prior.reset_index().rename(columns={'index': '변수'}), on='변수'))

ws5 = wb.create_sheet('Target측정근거')
measure = pd.DataFrame([
    ('문항', '문96. 지난 1년 동안 다음과 같은 건강상 문제가 있었습니까? — F. 전신 피로'),
    ('응답 보기', '① 있다 ② 없다 ⑧ 모름/무응답 ⑨ 거절 (Target: 있다=1, 없다=0, 모름·거절 제외)'),
    ('후속 문항', '문96-1-F. 일 때문에 발생했습니까? — 보건의료 교대근무자 전신피로 응답자의 ' + f'{work_related.mean():.0%}' + '가 그렇다'),
    ('변수명', 'heal_prob6 (후속 heal_prob6_1)'),
    ('조사 방식', '전문면접원 가구방문 1:1 면접조사, 태블릿 전자설문(TAPI) — 98.9% 현장면접'),
    ('측정 성격', '자기보고 단일 문항·이분형. 생체/객관적 측정(액티그래피, 반응시간 등) 없음'),
    ('타당성 확인', '같은 조사의 결근일수·아파도출근·주관적건강·WHO-5·퇴근시기진맥진과 비교 (아래 표)'),
], columns=['항목', '내용'])
write_table(ws5, measure)
ws5.column_dimensions['A'].width = 14; ws5.column_dimensions['B'].width = 95
r0 = len(measure) + 3
ws5.cell(row=r0, column=1, value=f'보건의료 교대근무자 (n={len(df)})').font = BOLD
write_table(ws5, valid_health.reset_index(), start_row=r0 + 1)
r1 = r0 + len(valid_health) + 3
ws5.cell(row=r1, column=1, value=f'참고: 전체 취업자 (n={len(all_workers):,})').font = BOLD
write_table(ws5, valid_all.reset_index(), start_row=r1 + 1)
r2 = r1 + len(valid_all) + 3
ws5.cell(row=r2, column=1, value='민감도 분석: 대체 Target(퇴근 시 기진맥진 항상/대부분) 로지스틱 회귀').font = BOLD
write_table(ws5, sens.reset_index(), start_row=r2 + 1)

OUT = '조별데이터_보건의료교대근무_전신피로.xlsx'
wb.save(OUT)
print('저장 완료:', OUT)
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass

## 참고문헌
- 한국산업안전보건공단 산업안전보건연구원. 제7차(2023년) 근로환경조사 원시자료, 코드북, 자료이용설명서. 공공데이터포털(data.go.kr).
- 한국 교대근무자의 피로에 대한 탐색적 조사 연구: 5차 근로환경조사를 이용하여. 한국웰니스학회지. https://www.dbpia.co.kr/journal/articleDetail?nodeId=NODE09271240